# Notebook 1: EfficientNetB3 + BERT Cross-Attention — Training + Analysis
Konfigurasi identik dengan baseline, ditambah analisis: GradCAM, t-SNE, confusion matrix, threshold curves, attention visualization.

In [ ]:
!pip install -q iterative-stratification grad-cam

In [ ]:
import os, random, warnings
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import BertTokenizer, BertModel
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import classification_report, f1_score, multilabel_confusion_matrix
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from torchmetrics.classification import MultilabelF1Score
from tqdm.auto import tqdm
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

warnings.filterwarnings('ignore')

GENRE_COLS = [
    'action', 'adventure', 'animation', 'comedy', 'crime',
    'drama', 'family', 'fantasy', 'horror', 'musical',
    'mystery', 'romance', 'scifi', 'thriller',
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Data Utilities (identik baseline)

In [ ]:
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace('_', ' ')
            if s.lower() != word.lower():
                synonyms.add(s)
    return list(synonyms)

def augment_text(text, n_replace=1):
    words = word_tokenize(text)
    candidate_indices = [i for i, w in enumerate(words) if get_synonyms(w)]
    if not candidate_indices:
        return text
    for idx in random.sample(candidate_indices, min(n_replace, len(candidate_indices))):
        syns = get_synonyms(words[idx])
        if syns:
            words[idx] = random.choice(syns)
    return ' '.join(words)

def stratified_train_val_split(df, val_ratio=0.2, random_state=42):
    y = df[GENRE_COLS].values.astype('float32')
    mss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=random_state)
    train_idx, val_idx = next(mss.split(df, y))
    df_train = df.iloc[train_idx].copy().reset_index(drop=True)
    df_val   = df.iloc[val_idx].copy().reset_index(drop=True)
    print(f'[Split] Train: {len(df_train)} | Val: {len(df_val)}')
    return df_train, df_val

def targeted_oversample(df_train, minority_threshold=15, ov_factor=5, random_state=42):
    counts = df_train[GENRE_COLS].sum()
    minority_genres = counts[counts < minority_threshold].index.tolist()
    if not minority_genres:
        return df_train.copy()
    minority_rows = df_train[df_train[minority_genres].sum(axis=1) > 0].copy()
    df_balanced = pd.concat([df_train] + [minority_rows] * ov_factor, ignore_index=True)
    df_balanced = df_balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)
    print(f'[Oversample] {len(df_train)} → {len(df_balanced)} baris')
    return df_balanced

EFFICIENTNET_NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def build_train_transform(image_size=300):
    return transforms.Compose([
        transforms.RandomResizedCrop((image_size, image_size), scale=(0.85, 1.0), ratio=(0.85, 1.15)),
        transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10, hue=0.00),
        transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
        transforms.RandomHorizontalFlip(p=0),
        transforms.ToTensor(),
        EFFICIENTNET_NORMALIZE,
    ])

def build_base_transform(image_size=300):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        EFFICIENTNET_NORMALIZE,
    ])

In [ ]:
class GenreDataset(Dataset):
    def __init__(self, dataframe, img_dir, tokenizer, max_text_len=32,
                 img_transform=None, image_size=300):
        self.filenames    = dataframe['filename'].values
        self.img_dir      = img_dir
        self.titles       = dataframe['title'].values
        self.labels       = dataframe[GENRE_COLS].values.astype('float32')
        self.tokenizer    = tokenizer
        self.max_text_len = max_text_len
        self.img_transform = img_transform or build_base_transform(image_size)
        self.image_size   = image_size

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.titles[idx]),
            padding='max_length', truncation=True,
            max_length=self.max_text_len, return_tensors='pt'
        )
        image = Image.open(os.path.join(self.img_dir, self.filenames[idx])).convert('RGB')
        pixel_values = self.img_transform(image)
        return {
            'input_ids'     : encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'pixel_values'  : pixel_values,
            'labels'        : torch.tensor(self.labels[idx], dtype=torch.float32),
            'filename'      : self.filenames[idx],
            'title'         : str(self.titles[idx]),
        }


class GenreDataModule(pl.LightningDataModule):
    def __init__(self, config, tokenizer, train_transform=None):
        super().__init__()
        self.config = config
        self.tokenizer = tokenizer
        self.train_transform = train_transform

    def setup(self, stage=None):
        def make_ctx(folder, ann):
            files_ = os.listdir(folder)
            df = pd.read_excel(ann, index_col=None)
            return df[df['filename'].isin(files_)].copy()

        cfg = self.config
        if stage in ('fit', None):
            df_raw = make_ctx(cfg['train_img_dir'], cfg['ann_path'])
            df_train_split, df_val_split = stratified_train_val_split(df_raw, val_ratio=cfg['val_ratio'])
            df_train = targeted_oversample(df_train_split, cfg['minority_threshold'], cfg['oversample_factor'])

            train_labels = df_train[GENRE_COLS].values.astype('float32')
            pos = train_labels.sum(0)
            self.pos_weight = torch.tensor((len(train_labels) - pos) / (pos + 1e-5), dtype=torch.float32)

            self.train_dataset = GenreDataset(df_train, cfg['train_img_dir'], self.tokenizer,
                                              cfg['max_text_len'], self.train_transform, cfg['image_size'])
            self.val_dataset   = GenreDataset(df_val_split, cfg['train_img_dir'], self.tokenizer,
                                              cfg['max_text_len'], build_base_transform(cfg['image_size']), cfg['image_size'])
            self.df_val = df_val_split  # simpan untuk analisis

        if stage in ('test', None):
            df_test = make_ctx(cfg['test_img_dir'], cfg['ann_path'])
            self.test_dataset = GenreDataset(df_test, cfg['test_img_dir'], self.tokenizer,
                                             cfg['max_text_len'], build_base_transform(cfg['image_size']), cfg['image_size'])
            self.df_test = df_test

    def train_dataloader(self):
        return DataLoader(self.train_dataset, self.config['batch_size'], shuffle=True,
                          num_workers=self.config['num_workers'], pin_memory=True, drop_last=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, self.config['batch_size'], shuffle=False,
                          num_workers=self.config['num_workers'], pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, self.config['batch_size'], shuffle=False,
                          num_workers=self.config['num_workers'], pin_memory=True)

## 2. Model (identik baseline)

In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        neg_p = (1 - p + self.clip).clamp(max=1.0)
        pos_loss = targets * torch.log(p.clamp(min=self.eps))
        neg_loss = (1 - targets) * torch.log(neg_p.clamp(min=self.eps))
        loss = -(torch.pow(1 - p, self.gamma_pos) * pos_loss +
                 torch.pow(1 - neg_p, self.gamma_neg) * neg_loss)
        return loss.mean()


class MultimodalLightningModel(pl.LightningModule):
    """
    EfficientNetB3 (visual) + BERT (text) → Cross-Attention Fusion → Classifier.
    Identik dengan baseline, dengan tambahan metode extract_features() untuk analisis.
    """

    def __init__(self, config):
        super().__init__()
        self.save_hyperparameters()

        self.bert = BertModel.from_pretrained(config['bert_name'], output_hidden_states=True)
        text_dim  = self.bert.config.hidden_size  # 768

        eff = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.efficientnet_features = eff.features
        self.efficientnet_avgpool  = nn.AdaptiveAvgPool2d(1)
        img_dim = 1536

        self.img_proj    = nn.Linear(img_dim, text_dim)
        self.cross_attn  = nn.MultiheadAttention(embed_dim=text_dim, num_heads=8, batch_first=True)
        self.layer_norm  = nn.LayerNorm(text_dim)

        self.classifier = nn.Sequential(
            nn.Linear(text_dim * 3, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(1024, config['num_classes']),
        )

        self.criterion      = nn.BCEWithLogitsLoss()
        self.current_phase  = 1
        self.val_f1         = MultilabelF1Score(num_labels=config['num_classes'], average='macro')

    def update_loss_weights(self, pos_weight_tensor):
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    def _extract_image_feat(self, pixel_values):
        x = self.efficientnet_features(pixel_values)
        x = self.efficientnet_avgpool(x).flatten(1)
        return self.img_proj(x)

    def _extract_text_feat(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return torch.stack([layer[:, 0, :] for layer in out.hidden_states[-4:]], dim=0).mean(0)

    def forward(self, input_ids, attention_mask, pixel_values):
        text_feat  = self._extract_text_feat(input_ids, attention_mask)
        image_feat = self._extract_image_feat(pixel_values)
        text_q  = text_feat.unsqueeze(1)
        image_k = image_feat.unsqueeze(1)
        attn_out, _ = self.cross_attn(text_q, image_k, image_k)
        cross_feat  = self.layer_norm(text_q + attn_out).squeeze(1)
        combined = torch.cat([text_feat, image_feat, cross_feat], dim=1)
        return self.classifier(combined)

    def extract_features(self, input_ids, attention_mask, pixel_values):
        """Return (combined_feat, text_feat, image_feat, cross_feat) untuk analisis."""
        text_feat  = self._extract_text_feat(input_ids, attention_mask)
        image_feat = self._extract_image_feat(pixel_values)
        text_q  = text_feat.unsqueeze(1)
        image_k = image_feat.unsqueeze(1)
        attn_out, attn_weights = self.cross_attn(text_q, image_k, image_k)
        cross_feat = self.layer_norm(text_q + attn_out).squeeze(1)
        combined   = torch.cat([text_feat, image_feat, cross_feat], dim=1)
        return combined, text_feat, image_feat, cross_feat, attn_weights

    def training_step(self, batch, batch_idx):
        logits = self(batch['input_ids'], batch['attention_mask'], batch['pixel_values'])
        loss   = self.criterion(logits, batch['labels'])
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch['input_ids'], batch['attention_mask'], batch['pixel_values'])
        loss   = self.criterion(logits, batch['labels'])
        self.val_f1(torch.sigmoid(logits), batch['labels'].long())
        self.log_dict({'val_loss': loss, 'val_f1': self.val_f1}, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        cfg = self.hparams.config
        if self.current_phase == 1:
            self.freeze_backbones()
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, self.parameters()),
                lr=cfg['lr_phase1'], weight_decay=cfg['weight_decay']
            )
            # Phase 1: CosineAnnealingLR
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cfg['epochs_phase1'],
                eta_min=cfg['lr_phase1_min'],
            )
            return {'optimizer': optimizer,
                    'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}}
        self.unfreeze_backbones()
        # Phase 2: backbone dan head punya LR terpisah
        optimizer = torch.optim.AdamW([
            {'params': self.bert.parameters(),                   'lr': cfg['lr_backbone']},
            {'params': self.efficientnet_features.parameters(),  'lr': cfg['lr_backbone']},
            {'params': self.img_proj.parameters(),               'lr': cfg['lr_classifier']},
            {'params': self.cross_attn.parameters(),             'lr': cfg['lr_classifier']},
            {'params': self.layer_norm.parameters(),             'lr': cfg['lr_classifier']},
            {'params': self.classifier.parameters(),             'lr': cfg['lr_classifier']},
        ], weight_decay=cfg['weight_decay'])
        # Phase 2: OneCycleLR
        steps = len(self.trainer.datamodule.train_dataloader())
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[cfg['lr_backbone']] * 2 + [cfg['lr_classifier']] * 4,
            epochs=cfg['epochs_phase2'],
            steps_per_epoch=steps,
            pct_start=0.15,
            anneal_strategy='cos',
            div_factor=10.0,
            final_div_factor=100.0,
        )
        return {'optimizer': optimizer,
                'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}

    def set_phase(self, p): self.current_phase = p

    def freeze_backbones(self):
        for m in [self.bert, self.efficientnet_features]:
            for p in m.parameters(): p.requires_grad = False
            m.eval()
        print('[Model] Backbone FROZEN')

    def unfreeze_backbones(self):
        for m in [self.bert, self.efficientnet_features]:
            for p in m.parameters(): p.requires_grad = True
            m.train()
        print('[Model] Backbone UNFROZEN')

## 3. Config & Training

In [ ]:
CONFIG = {
    'ann_path'        : '/kaggle/input/datasets/nafalrust/compvi-polosan/Datasets/genre.xlsx',
    'train_img_dir'   : '/kaggle/input/datasets/nafalrust/compvi-polosan/Datasets/images/train',
    'test_img_dir'    : '/kaggle/input/datasets/nafalrust/compvi-polosan/Datasets/images/test',
    'val_ratio'       : 0.2,
    'minority_threshold': 15,
    'oversample_factor' : 5,
    'random_state'    : 42,
    'batch_size'      : 16,
    'max_text_len'    : 16,
    'num_workers'     : 1,
    'image_size'      : 300,
    'bert_name'       : 'bert-base-uncased',
    'dropout_rate'    : 0.3,
    'num_classes'     : 14,
    'epochs_phase1'   : 30,
    'epochs_phase2'   : 60,
    'lr_phase1'       : 5e-4,
    'lr_phase1_min'   : 1e-6,
    'lr_backbone'     : 1e-5,
    'lr_classifier'   : 5e-5,
    'weight_decay'    : 0.01,
    'steps_per_epoch' : -1,
}

tokenizer   = BertTokenizer.from_pretrained(CONFIG['bert_name'])
data_module = GenreDataModule(CONFIG, tokenizer, build_train_transform(CONFIG['image_size']))
data_module.setup(stage='fit')

In [ ]:
# Phase 1: Frozen backbone
model = MultimodalLightningModel(config=CONFIG)
model.update_loss_weights(data_module.pos_weight)
model.set_phase(1)

ckpt_p1  = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1,
                            dirpath='checkpoints/', filename='best_model_phase1')
trainer_p1 = pl.Trainer(
    max_epochs=CONFIG['epochs_phase1'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
    callbacks=[ckpt_p1], logger=CSVLogger('logs', 'phase1')
)
trainer_p1.fit(model, datamodule=data_module)

In [ ]:
# Phase 2: Fine-tuning end-to-end
model = MultimodalLightningModel.load_from_checkpoint(ckpt_p1.best_model_path, strict=False)
model.update_loss_weights(data_module.pos_weight)
model.set_phase(2)

ckpt_p2  = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1,
                            dirpath='checkpoints/', filename='best_model_phase2')
trainer_p2 = pl.Trainer(
    max_epochs=CONFIG['epochs_phase2'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
    callbacks=[ckpt_p2], logger=CSVLogger('logs', 'phase2')
)
trainer_p2.fit(model, datamodule=data_module)

In [ ]:
# Loss Curves
def plot_loss_curve(log_version_path, phase_name):
    df = pd.read_csv(f'{log_version_path}/metrics.csv')
    agg = df.groupby('epoch')[['train_loss', 'val_loss']].mean()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(agg.index, agg['train_loss'], 'b-o', label='Train Loss')
    ax.plot(agg.index, agg['val_loss'],   'r-s', label='Val Loss')
    ax.set(title=f'Loss Curve — {phase_name}', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout(); plt.show()

plot_loss_curve('/kaggle/working/logs/phase1/version_0', 'Phase 1')
plot_loss_curve('/kaggle/working/logs/phase2/version_0', 'Phase 2')

## 4. Threshold Tuning (identik baseline)

In [ ]:
best_model = MultimodalLightningModel.load_from_checkpoint(ckpt_p2.best_model_path, strict=False)
best_model.eval().to(device)

val_loader = data_module.val_dataloader()

def collect_probs(model, loader, device):
    all_probs, all_targets, all_feats = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Collecting'):
            ids = batch['input_ids'].to(device)
            msk = batch['attention_mask'].to(device)
            pxl = batch['pixel_values'].to(device)
            logits = model(ids, msk, pxl)
            combined, *_ = model.extract_features(ids, msk, pxl)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(batch['labels'].numpy())
            all_feats.append(combined.cpu().numpy())
    return np.vstack(all_probs), np.vstack(all_targets), np.vstack(all_feats)

val_probs, val_true, val_feats = collect_probs(best_model, val_loader, device)

def find_best_thresholds(y_probs, y_true, threshold_range=None):
    if threshold_range is None:
        threshold_range = np.arange(0.05, 0.96, 0.01)
    best_thr, best_f1 = {}, {}
    for i, g in enumerate(GENRE_COLS):
        scores = [(f1_score(y_true[:, i], (y_probs[:, i] > t).astype(int), zero_division=0), t)
                  for t in threshold_range]
        bscore, bt = max(scores)
        best_thr[g] = round(float(bt), 2)
        best_f1[g]  = round(float(bscore), 4)
    thr_arr = np.array([best_thr[g] for g in GENRE_COLS])
    y_pred_opt  = (y_probs > thr_arr).astype(int)
    print(f'Macro F1 (0.5):     {f1_score(y_true, (y_probs > 0.5).astype(int), average="macro", zero_division=0):.4f}')
    print(f'Macro F1 (optimal): {f1_score(y_true, y_pred_opt, average="macro", zero_division=0):.4f}')
    return best_thr, best_f1

BEST_THRESHOLDS, f1_per_genre_val = find_best_thresholds(val_probs, val_true)
print(BEST_THRESHOLDS)

In [ ]:
#  Threshold vs F1 plot
thr_range = np.arange(0.05, 0.96, 0.01)
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for i, (genre, ax) in enumerate(zip(GENRE_COLS, axes.flatten())):
    f1s = [f1_score(val_true[:, i], (val_probs[:, i] > t).astype(int), zero_division=0) for t in thr_range]
    ax.plot(thr_range, f1s, 'steelblue')
    ax.axvline(BEST_THRESHOLDS[genre], color='r', ls='--', label=f"t={BEST_THRESHOLDS[genre]}")
    ax.set(title=genre.capitalize(), xlabel='Threshold', ylabel='F1', xlim=(0,1), ylim=(0,1))
    ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.4)
for ax in axes.flatten()[len(GENRE_COLS):]:
    ax.set_visible(False)
plt.suptitle('Threshold vs F1 per Genre (Val Set)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('threshold_f1.png', dpi=150); plt.show()

## 5. Test Evaluation + TTA

In [ ]:
data_module.setup(stage='test')
test_loader = data_module.test_dataloader()

tta_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
])

def evaluate_with_tta(model, loader, device, genre_thresholds, tta_steps=5):
    model.eval().to(device)
    all_probs, all_targets, all_feats = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'TTA={tta_steps}'):
            ids = batch['input_ids'].to(device)
            msk = batch['attention_mask'].to(device)
            pxl = batch['pixel_values'].to(device)
            step_probs = []
            for s in range(tta_steps):
                aug = pxl if s == 0 else tta_transforms(pxl)
                step_probs.append(torch.sigmoid(model(ids, msk, aug)).unsqueeze(0))
            avg = torch.cat(step_probs).mean(0)
            all_probs.append(avg.cpu().numpy())
            all_targets.append(batch['labels'].numpy())
            combined, *_ = model.extract_features(ids, msk, pxl)
            all_feats.append(combined.cpu().numpy())

    y_probs  = np.vstack(all_probs)
    y_true   = np.vstack(all_targets)
    y_feats  = np.vstack(all_feats)
    thr_arr  = np.array([genre_thresholds[g] for g in GENRE_COLS])
    y_pred   = (y_probs > thr_arr).astype(int)
    print(classification_report(y_true, y_pred, target_names=GENRE_COLS, zero_division=0))
    print(f'Macro F1: {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
    print(f'Micro F1: {f1_score(y_true, y_pred, average="micro", zero_division=0):.4f}')
    return y_probs, y_pred, y_true, y_feats

y_probs_test, y_pred_test, y_true_test, test_feats = evaluate_with_tta(
    best_model, test_loader, device, BEST_THRESHOLDS, tta_steps=5
)

In [ ]:
# Save Predictions to CSV (binary + probabilities)
GENRE_COLS = [
    'action', 'adventure', 'animation', 'comedy', 'crime',
    'drama', 'family', 'fantasy', 'horror', 'musical',
    'mystery', 'romance', 'scifi', 'thriller',
]

df_test = data_module.df_test.copy()
filenames = pd.DataFrame({'filename': df_test['filename'].values})

#  1. Binary predictions (0/1) 
pred_bin = pd.DataFrame(y_pred_test, columns=GENRE_COLS)
pred_bin['predicted_genres'] = pred_bin[GENRE_COLS].apply(
    lambda row: ', '.join([g for g in GENRE_COLS if row[g] == 1]), axis=1
)
result_bin = pd.concat([filenames, pred_bin], axis=1)
out_bin = '/kaggle/working/predictions_test_binary.csv'
result_bin.to_csv(out_bin, index=False)
print(f'Saved binary predictions ({len(result_bin)} rows) → {out_bin}')
display(result_bin.head())

#  2. Probability scores per genre 
pred_prob = pd.DataFrame(
    np.round(y_probs_test, 4), columns=[f'prob_{g}' for g in GENRE_COLS]
)
result_prob = pd.concat([filenames, pred_prob], axis=1)
out_prob = '/kaggle/working/predictions_test_probs.csv'
result_prob.to_csv(out_prob, index=False)
print(f'Saved probability predictions ({len(result_prob)} rows) → {out_prob}')
display(result_prob.head())


## 6. Analysis — Multilabel Confusion Matrix

In [ ]:
#  Per-genre confusion matrix (TP/FP/FN/TN) ─
mcm = multilabel_confusion_matrix(y_true_test, y_pred_test)

fig, axes = plt.subplots(4, 4, figsize=(14, 12))
for i, (genre, ax) in enumerate(zip(GENRE_COLS, axes.flatten())):
    cm = mcm[i]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'],
                cbar=False, linewidths=0.5)
    ax.set_title(genre.capitalize(), fontsize=10, fontweight='bold')
for ax in axes.flatten()[len(GENRE_COLS):]:
    ax.set_visible(False)
plt.suptitle('Per-Genre Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

## 7. Analysis — t-SNE Feature Visualization

In [ ]:
#  t-SNE on combined multimodal features
# Gabungkan val + test features untuk populasi lebih besar
all_feats   = np.vstack([val_feats, test_feats])
all_labels  = np.vstack([val_true,  y_true_test])

# PCA 50-dim dulu untuk efisiensi t-SNE
pca = PCA(n_components=50, random_state=42)
feats_pca = pca.fit_transform(all_feats)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}')

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42, n_jobs=-1)
feats_2d = tsne.fit_transform(feats_pca)
print('t-SNE done.')

In [ ]:
#  t-SNE colored per genre
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for i, (genre, ax) in enumerate(zip(GENRE_COLS, axes.flatten())):
    mask_pos = all_labels[:, i] == 1
    ax.scatter(feats_2d[~mask_pos, 0], feats_2d[~mask_pos, 1],
               c='lightgray', s=8, alpha=0.4, label='Other')
    ax.scatter(feats_2d[mask_pos, 0],  feats_2d[mask_pos, 1],
               c='crimson', s=20, alpha=0.8, label=genre.capitalize())
    ax.set(title=genre.capitalize(), xticks=[], yticks=[])
    ax.legend(fontsize=7, loc='upper right')
for ax in axes.flatten()[len(GENRE_COLS):]:
    ax.set_visible(False)
plt.suptitle('t-SNE of Multimodal Features per Genre', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('tsne_per_genre.png', dpi=150); plt.show()

In [ ]:
#  t-SNE: label count per sample (heatmap style)
label_count = all_labels.sum(axis=1)
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(feats_2d[:, 0], feats_2d[:, 1],
                c=label_count, cmap='plasma', s=12, alpha=0.7)
plt.colorbar(sc, ax=ax, label='# Active Genres')
ax.set(title='t-SNE — Sample Colored by #Genres', xticks=[], yticks=[])
plt.tight_layout(); plt.savefig('tsne_label_count.png', dpi=150); plt.show()

## 8. Analysis — GradCAM

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Wrapper agar GradCAM bisa dipakai pada image-only path
class ImageOnlyWrapper(nn.Module):
    """
    Wrapper yang menerima pixel_values saja.
    Text input di-cache dari batch pertama (frozen selama GradCAM).
    """
    def __init__(self, model, fixed_input_ids, fixed_attention_mask):
        super().__init__()
        self.model              = model
        self.fixed_input_ids    = fixed_input_ids
        self.fixed_attention_mask = fixed_attention_mask

    def forward(self, pixel_values):
        return self.model(
            self.fixed_input_ids,
            self.fixed_attention_mask,
            pixel_values,
        )

def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406])
    std  = torch.tensor([0.229, 0.224, 0.225])
    t = tensor.clone().cpu()
    for c in range(3):
        t[c] = t[c] * std[c] + mean[c]
    return t.clamp(0, 1).permute(1, 2, 0).numpy()


def run_gradcam_analysis(model, data_module, device, genre_thresholds, n_samples=6):
    """
    Tampilkan GradCAM untuk setiap genre yang diprediksi positif.
    Target layer: blok terakhir EfficientNetB3 = efficientnet_features[-1].
    """
    model.eval().to(device)
    dataset   = data_module.test_dataset
    thr_arr   = np.array([genre_thresholds[g] for g in GENRE_COLS])

    # Ambil beberapa sample acak
    indices   = random.sample(range(len(dataset)), n_samples)

    target_layer = [model.efficientnet_features[-1]]

    for idx in indices:
        sample    = dataset[idx]
        pxl       = sample['pixel_values'].unsqueeze(0).to(device)     # (1,3,H,W)
        ids       = sample['input_ids'].unsqueeze(0).to(device)
        msk       = sample['attention_mask'].unsqueeze(0).to(device)
        true_lbl  = sample['labels'].numpy()
        title     = sample['title']

        with torch.no_grad():
            logits = model(ids, msk, pxl)
        probs  = torch.sigmoid(logits).cpu().numpy()[0]
        pred   = (probs > thr_arr).astype(int)

        # Genre yang diprediksi positif
        pred_genres = [GENRE_COLS[i] for i in range(14) if pred[i] == 1]
        if not pred_genres:
            pred_genres = [GENRE_COLS[np.argmax(probs)]]  # fallback: genre paling yakin

        wrapper = ImageOnlyWrapper(model, ids, msk)

        rgb_img = denormalize(sample['pixel_values'])

        ncols = min(len(pred_genres) + 1, 5)
        fig, axes = plt.subplots(1, ncols, figsize=(ncols * 3, 3.5))
        if ncols == 1: axes = [axes]

        axes[0].imshow(rgb_img)
        true_str = ', '.join([GENRE_COLS[i] for i in range(14) if true_lbl[i]])
        axes[0].set_title(f'"{title}"\nTrue: {true_str}', fontsize=8)
        axes[0].axis('off')

        for j, genre in enumerate(pred_genres[:ncols-1]):
            genre_idx = GENRE_COLS.index(genre)
            with GradCAM(model=wrapper, target_layers=target_layer) as cam:
                targets   = [ClassifierOutputTarget(genre_idx)]
                grayscale = cam(input_tensor=pxl, targets=targets)
            cam_img = show_cam_on_image(rgb_img, grayscale[0], use_rgb=True, colormap=9)
            axes[j+1].imshow(cam_img)
            axes[j+1].set_title(f'GradCAM: {genre}\n(prob={probs[genre_idx]:.2f})', fontsize=8)
            axes[j+1].axis('off')

        plt.suptitle('GradCAM Analysis', fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'gradcam_{title[:20].replace(" ","_")}.png', dpi=120)
        plt.show()

run_gradcam_analysis(best_model, data_module, device, BEST_THRESHOLDS, n_samples=8)

## 9. Analysis — Attention Weight Visualization (Cross-Attention)

In [ ]:
#  Cross-attention weight per sample
# Cross-attention kita adalah single-query single-key (text[CLS] vs image feat),
# sehingga attn_weights shape = (B, num_heads, 1, 1) → skalar per head.
# Visualisasi: per-sample attention score per head.

def collect_attention_weights(model, loader, device, max_samples=200):
    model.eval().to(device)
    all_attn, all_labels = [], []
    n = 0
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            msk = batch['attention_mask'].to(device)
            pxl = batch['pixel_values'].to(device)
            _, _, _, _, attn_w = model.extract_features(ids, msk, pxl)
            # attn_w: (B, num_heads, 1, 1)
            # attn_w shape: (B, num_heads, 1, 1) → reshape to (B, num_heads)
            aw = attn_w.reshape(attn_w.shape[0], -1).cpu().numpy()
            all_attn.append(aw)
            all_labels.append(batch['labels'].numpy())
            n += len(ids)
            if n >= max_samples:
                break
    return np.vstack(all_attn), np.vstack(all_labels)

attn_weights, attn_labels = collect_attention_weights(best_model, val_loader, device)

# Rata-rata attention per genre
fig, axes = plt.subplots(4, 4, figsize=(14, 11))
for i, (genre, ax) in enumerate(zip(GENRE_COLS, axes.flatten())):
    mask = attn_labels[:, i] == 1
    if mask.sum() == 0:
        ax.set_visible(False); continue
    mean_attn_pos = attn_weights[mask].mean(0)    # (8,)
    mean_attn_neg = attn_weights[~mask].mean(0)
    x = np.arange(attn_weights.shape[1])
    ax.bar(x - 0.2, mean_attn_pos, 0.4, label='Positive', color='steelblue')
    ax.bar(x + 0.2, mean_attn_neg, 0.4, label='Negative', color='salmon')
    ax.set(title=genre.capitalize(), xlabel='Head', ylabel='Avg Attn')
    ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.3)
for ax in axes.flatten()[len(GENRE_COLS):]:
    ax.set_visible(False)
plt.suptitle('Cross-Attention Weight per Head (Pos vs Neg Samples)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('attention_weights.png', dpi=150); plt.show()

## 10. Analysis — Class-wise Performance Summary

In [ ]:
from sklearn.metrics import precision_score, recall_score

thr_arr = np.array([BEST_THRESHOLDS[g] for g in GENRE_COLS])
y_pred  = (y_probs_test > thr_arr).astype(int)

summary = pd.DataFrame({
    'Genre'    : GENRE_COLS,
    'Support'  : y_true_test.sum(0).astype(int),
    'Threshold': [BEST_THRESHOLDS[g] for g in GENRE_COLS],
    'Precision': precision_score(y_true_test, y_pred, average=None, zero_division=0).round(3),
    'Recall'   : recall_score(y_true_test, y_pred, average=None, zero_division=0).round(3),
    'F1'       : [f1_score(y_true_test[:, i], y_pred[:, i], zero_division=0) for i in range(14)],
}).sort_values('F1', ascending=False).reset_index(drop=True)

print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(GENRE_COLS))
ordered = summary.sort_values('Genre')
ax.bar(x - 0.25, ordered['Precision'], 0.25, label='Precision', color='steelblue')
ax.bar(x,        ordered['Recall'],    0.25, label='Recall',    color='darkorange')
ax.bar(x + 0.25, ordered['F1'],        0.25, label='F1',        color='seagreen')
ax.set(xticks=x, xticklabels=ordered['Genre'], title='Per-Genre Precision / Recall / F1 (Test Set)')
ax.tick_params(axis='x', rotation=45)
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(True, ls='--', alpha=0.3)
plt.tight_layout(); plt.savefig('per_genre_metrics.png', dpi=150); plt.show()

## 11. Analysis — Probability Calibration (Reliability Diagram)

In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(4, 4, figsize=(14, 11))
for i, (genre, ax) in enumerate(zip(GENRE_COLS, axes.flatten())):
    if y_true_test[:, i].sum() == 0:
        ax.set_visible(False); continue
    try:
        frac_pos, mean_pred = calibration_curve(y_true_test[:, i], y_probs_test[:, i], n_bins=10)
        ax.plot(mean_pred, frac_pos, 's-', color='steelblue', label='Model')
        ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Perfect')
    except Exception:
        pass
    ax.set(title=genre.capitalize(), xlabel='Mean Predicted Prob', ylabel='Fraction Positive',
           xlim=(0,1), ylim=(0,1))
    ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.4)
for ax in axes.flatten()[len(GENRE_COLS):]:
    ax.set_visible(False)
plt.suptitle('Calibration / Reliability Diagram per Genre', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('calibration.png', dpi=150); plt.show()